# ============================================================
# LOAN DECISION ENGINE v2
# Extended Hybrid Decision Engine
#
# Architecture:
#   User Input
#     → Extract multiple variables (amount + credit_score)
#     → Parse AND conditions
#     → Evaluate rule logic
#     → Apply priority strategy
#     → Select best rule
#     → LLM explanation
#     → Structured output
# ============================================================


In [57]:
# ============================================================
# SECTION 0 — Setup & Imports
# ============================================================

import csv
import json
import re
import torch
import pandas as pd
import numpy as np
from typing import Dict, Any, Tuple, Optional, List

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install bitsandbytes accelerate

In [59]:
# ============================================================
# SECTION 1 — Model Loading
# Load Phi-3.5 Mini Instruct in 4-bit quantization
# ============================================================

# Change these paths to match your environment
model_path = "/content/drive/MyDrive/Phi_3_5_mini_instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    local_files_only=True
)

print("Model & tokenizer loaded")



Device: cuda


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Model & tokenizer loaded


In [60]:
# ============================================================
# SECTION 2 — Policy Rules CSV
# Define and write loan policy rules to CSV
#
# Columns:
#   section          — domain (e.g. Loan)
#   policy_name      — short rule name
#   rule_description — human-readable description
#   condition        — evaluable condition string (supports AND)
#   approval_required— decision outcome
#   risk_level       — Low / Medium / High
# ============================================================

loan_rules_v2 = [
    ["section", "policy_name", "rule_description", "condition", "approval_required", "risk_level"],
    ["Loan", "Small Loan",           "Small loan auto approved",           "amount <= 5000",                              "APPROVED",             "Low"],
    ["Loan", "Medium Loan",          "Medium loan review",                 "5000 < amount <= 20000",                      "MANUAL_REVIEW",        "Medium"],
    ["Loan", "High Loan Low Credit", "High loan with low credit rejected", "amount > 20000 AND credit_score < 700",       "REJECTED",             "High"],
    ["Loan", "Very Low Credit",      "Very low credit score rejection",    "credit_score < 600",                          "REJECTED",             "High"],
]

loan_csv_v2 = "Loan_Policy_v2.csv"

with open(loan_csv_v2, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(loan_rules_v2)

print("Loan v2 CSV created.")

Loan v2 CSV created.


In [61]:
# ============================================================
# SECTION 3 — Policy DataFrame Loading
# Load CSV into DataFrame and build full_text column
# used for semantic embedding
# ============================================================

Policy_Csv = loan_csv_v2

df = pd.read_csv(Policy_Csv)

cols = ["section", "policy_name", "rule_description",
        "condition", "approval_required", "risk_level"]

df[cols] = df[cols].fillna("")

df["full_text"] = df.apply(lambda r: clean_text(
    f"{r['section']} | {r['policy_name']} | "
    f"{r['rule_description']} | {r['condition']} | "
    f"{r['approval_required']} | {r['risk_level']}"
), axis=1)

policy_texts = df["full_text"].tolist()
print("Loan v2 rules loaded:", len(policy_texts))
for t in policy_texts:
    print(" ", t)


Loan v2 rules loaded: 4
  Loan | Small Loan | Small loan auto approved | amount <= 5000 | APPROVED | Low
  Loan | Medium Loan | Medium loan review | 5000 < amount <= 20000 | MANUAL_REVIEW | Medium
  Loan | High Loan Low Credit | High loan with low credit rejected | amount > 20000 AND credit_score < 700 | REJECTED | High
  Loan | Very Low Credit | Very low credit score rejection | credit_score < 600 | REJECTED | High


In [62]:
# ============================================================
# SECTION 4 — Semantic Embedding Setup
# Encode all policy texts into vector embeddings
# for cosine similarity retrieval
# ============================================================

embedder = SentenceTransformer("all-MiniLM-L6-v2")
policy_emb = embedder.encode(
    policy_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Loan embeddings ready.")


def semantic_topk(query: str, k: int = 5):
    """
    Retrieve top-k most similar policies using cosine similarity.

    Encodes the user query into a vector, then computes dot-product
    similarity against all pre-encoded policy embeddings.
    Returns the k highest-scoring policies sorted by relevance.

    Args:
        query : Raw user question string.
        k     : Number of top candidates to return.

    Returns:
        List of dicts [{"score": float, "text": str}, ...]
        sorted by descending similarity score.
    """
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = (policy_emb @ q_emb.T).squeeze()

    idx = np.argsort(scores)[-k:][::-1]

    return [
        {"score": float(scores[i]), "text": policy_texts[i]}
        for i in idx
    ]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loan embeddings ready.


In [63]:
# ============================================================
# SECTION 5 — Signal Dictionary & Keyword Matching
# SIGNAL_DICT maps category labels to trigger keywords.
# Used by extract_signals() and rerank_with_signals()
# to boost policy candidates that match user intent.
# ============================================================

SIGNAL_DICT = {
    "amounts": {
        "small":  ["small", "little", "minor", "1000", "2000", "3000", "4000", "5000"],
        "medium": ["medium", "moderate", "12000", "15000", "8000", "9000", "10000"],
        "large":  ["large", "big", "huge", "25000", "30000", "50000", "100000"]
    },

    "credit": {
        "excellent": ["excellent credit", "great credit", "800", "850"],
        "good":      ["good credit", "700", "710", "720", "740", "750", "760"],
        "poor":      ["bad credit", "low credit", "poor credit", "580", "590", "600", "650", "680"]
    },

    "actions": {
        "apply":    ["apply", "applying", "application"],
        "borrow":   ["borrow", "borrowing", "loan", "request", "need"]
    }
}


def keyword_match(text, keywords):
    """
    Match a list of keywords against text using word boundaries.

    Uses regex word boundaries (\\b) to avoid false substring matches.
    For example, "low" will not match "elbow".

    Args:
        text     : Lowercased string to search in.
        keywords : List of keyword strings to look for.

    Returns:
        True if any keyword is found, False otherwise.
    """
    for kw in keywords:
        if re.search(rf"\b{re.escape(kw)}\b", text):
            return True
    return False


def extract_signals(q: str):
    """
    Extract keyword-based signals from the user query.

    Iterates over SIGNAL_DICT categories and labels.
    For each label whose keywords match the query,
    adds the label to the corresponding signal category list.

    Args:
        q : Raw user question string.

    Returns:
        Dict with one key per SIGNAL_DICT category,
        each mapped to a list of matched label strings.
        Example: {"amounts": ["large"], "credit": ["poor"], "actions": ["borrow"]}
    """
    ql = q.lower()
    signals = {category: [] for category in SIGNAL_DICT.keys()}

    for category, items in SIGNAL_DICT.items():
        for label, keywords in items.items():
            if keyword_match(ql, keywords):
                signals[category].append(label)

    return signals


def rerank_with_signals(cands, signals, bonus=0.08):
    """
    Boost candidate policy scores based on matched signals.

    For each candidate, adds a fixed bonus score for every
    signal label that appears in the policy text.
    Re-sorts candidates by updated score descending.

    Args:
        cands   : List of candidate dicts [{"score": float, "text": str}].
        signals : Output of extract_signals() — dict of matched labels.
        bonus   : Score bonus added per matched signal label (default 0.08).

    Returns:
        Re-sorted list of candidate dicts with updated scores.
    """
    out = []
    for c in cands:
        text_l = c["text"].lower()
        score = c["score"]

        for category, labels in signals.items():
            for label in labels:
                if label in text_l:
                    score += bonus

        out.append({"score": float(score), "text": c["text"]})

    out.sort(key=lambda x: x["score"], reverse=True)
    return out


In [64]:
# ============================================================
# SECTION 6 — Numeric Extraction & Condition Matching
# Core utilities to extract numbers from text and evaluate
# rule conditions without using eval()
# ============================================================

def extract_numbers(q: str):
    """
    Extract all numeric values from a user query string.

    Handles integers, decimals, and comma-formatted numbers.
    Used internally by extract_variables() for per-sentence parsing.

    Args:
        q : Raw query or sentence fragment string.

    Returns:
        List of floats parsed from the string.
        Example: "loan of 25,000" → [25000.0]
    """
    nums = re.findall(r"\d{1,9}(?:,\d{9})*(?:\.\d+)?", q)
    return [float(n.replace(",", "")) for n in nums]


def get_condition_from_rule_text(rule_text: str) -> str:
    """
    Extract the condition field from a pipe-separated rule text string.

    Expected format:
        section | policy_name | rule_description | condition | approval | risk

    Args:
        rule_text : Full pipe-separated policy text string.

    Returns:
        The condition string (4th field), or empty string if not found.
    """
    parts = [p.strip() for p in rule_text.split("|")]
    return parts[3].strip() if len(parts) >= 4 else ""


def match_numeric_condition(cond: str, value: float) -> bool:
    """
    Evaluate a single numeric condition string against a value.

    Supports two condition formats without using eval():
        - Chained range : 5000 < amount <= 20000
        - Simple        : amount > 20000  |  credit_score <= 600

    The variable name in the condition is treated as a placeholder
    and matched generically via \\w+.

    Args:
        cond  : Condition string (single clause, no AND).
        value : Numeric value to test against the condition.

    Returns:
        True if the condition is satisfied, False otherwise.
    """
    cond = cond.strip()
    if not cond:
        return False

    # chained: 1500 < amount <= 3000
    m = re.match(r"^\s*([0-9\.]+)\s*<\s*\w+\s*<=\s*([0-9\.]+)\s*$", cond)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
        return lo < value <= hi

    # simple: amount > 3000  OR payment_terms_days <= 60
    m = re.match(r"^\s*\w+\s*(<=|>=|<|>|==)\s*([0-9\.]+)\s*$", cond)
    if m:
        op, rhs = m.group(1), float(m.group(2))
        if op == "<=": return value <= rhs
        if op == ">=": return value >= rhs
        if op == "<":  return value < rhs
        if op == ">":  return value > rhs
        if op == "==": return value == rhs

    return False



In [65]:
# ============================================================
# SECTION 7 — Variable Extraction
# Extract named numeric variables (amount, credit_score)
# from user input using keyword-guided sentence scanning
# ============================================================

# Keyword map: variable name → trigger words to look for in question
VARIABLE_KEYWORDS = {
    "amount":       ["loan", "borrow", "request", "need", "apply", "amount"],
    "credit_score": ["credit", "score", "fico", "rating"]
}


def extract_variables(q: str) -> dict:
    """
    Extract named numeric variables from user input.

    Strategy:
        1. Split query into sentence fragments on punctuation and "and".
        2. For each variable, find the first sentence containing
           a known trigger keyword.
        3. Extract the first number from that sentence and assign
           it to the variable.

    Uses VARIABLE_KEYWORDS to map variable names to trigger words,
    and extract_numbers() for numeric parsing within each fragment.

    Args:
        q : Raw user question string.

    Returns:
        Dict mapping variable names to extracted float values.
        Example: {"amount": 25000.0, "credit_score": 680.0}
        Missing variables are simply absent from the dict.
    """
    ql = q.lower()
    result = {}

    # Split into sentences for localized keyword matching
    sentences = re.split(r"[.,;]|\band\b", ql)

    for var, keywords in VARIABLE_KEYWORDS.items():
        for sentence in sentences:
            if any(kw in sentence for kw in keywords):
                # Reuse extract_numbers on this sentence fragment
                nums = extract_numbers(sentence)
                if nums:
                    result[var] = nums[0]
                    break   # first matching sentence wins

    return result

In [66]:
# ============================================================
# SECTION 8 — AND Condition Matching & Priority Rule Selection
# Evaluate multi-variable AND conditions and select the
# highest-priority matching rule
# ============================================================

def match_and_condition(condition: str, variables: dict) -> bool:
    """
    Evaluate a condition string that may contain AND clauses.

    Splits the condition on AND, then evaluates each part
    independently using match_numeric_condition().
    All parts must be True for the overall condition to pass.

    Supports:
        amount <= 5000
        5000 < amount <= 20000
        amount > 20000 AND credit_score < 700
        credit_score < 600

    No eval() is used — safe manual evaluation only.

    Args:
        condition : Full condition string, may contain AND.
        variables : Dict of extracted variables e.g. {"amount": 25000, "credit_score": 680}.

    Returns:
        True if ALL condition parts are satisfied, False otherwise.
        Returns False if a referenced variable is missing from variables dict.
    """
    # Split on AND — gives list of individual conditions
    parts = [p.strip() for p in re.split(r"\bAND\b", condition, flags=re.IGNORECASE)]

    for part in parts:

        # Detect which variable this part refers to
        matched_var = None
        matched_val = None

        for var, value in variables.items():
            if var in part:
                matched_var = var
                matched_val = value
                break

        # Variable referenced in condition but not supplied by user → fail this part
        if matched_var is None:
            return False

        # Normalize: replace variable name with placeholder "amount"
        # so match_numeric_condition() (which uses \w+) can parse it
        normalized = re.sub(rf"\b{re.escape(matched_var)}\b", "amount", part)

        # Reuse match_numeric_condition from Section 6
        if not match_numeric_condition(normalized, matched_val):
            return False

    # All parts passed
    return True


# Risk priority weights — Higher number = higher priority
RISK_PRIORITY = {"High": 3, "Medium": 2, "Low": 1}


def apply_priority_rule_matching(candidates: list, variables: dict):
    """
    Evaluate all candidate rules and select the highest-priority match.

    For each candidate:
        1. Extracts the condition string from rule text.
        2. Checks if the condition contains numeric clauses.
        3. Evaluates AND conditions via match_and_condition().
        4. Collects all matching rules with their risk priority.

    Selection strategy:
        - Higher risk_level wins (High > Medium > Low).
        - Same risk level → higher semantic score wins.
        - No match → returns fallback=True.

    Args:
        candidates : List of candidate dicts from semantic search.
        variables  : Dict of extracted variables {"amount": ..., "credit_score": ...}.

    Returns:
        Tuple of (best_candidate, constraint_used, fallback):
            best_candidate  — the selected rule dict.
            constraint_used — True if a numeric condition was matched.
            fallback        — True if no rule matched (trigger NEED_MORE_INFORMATION).
    """
    if not variables:
        # No variables extracted — cannot evaluate any numeric condition
        return candidates[0], False, True

    matched_rules = []
    unmatched_rules = []

    for c in candidates:
        cond = get_condition_from_rule_text(c["text"])

        # Extract risk level from rule text (6th pipe-separated field)
        parts = [p.strip() for p in c["text"].split("|")]
        risk  = parts[5] if len(parts) >= 6 else "Low"

        has_numeric = bool(re.search(r"\w+\s*(<=|>=|<|>|==)\s*\d", cond)) or \
                      bool(re.search(r"\d+\s*<\s*\w+\s*<=\s*\d+", cond))

        if has_numeric:
            if match_and_condition(cond, variables):
                matched_rules.append((RISK_PRIORITY.get(risk, 0), c["score"], risk, c))
            else:
                unmatched_rules.append(c)
        else:
            unmatched_rules.append(c)

    if not matched_rules:
        # No rule matched — trigger fallback
        return candidates[0], False, True

    # Sort: highest risk priority first, then semantic score as tiebreaker
    matched_rules.sort(key=lambda x: (x[0], x[1]), reverse=True)

    _, _, _, best = matched_rules[0]
    return best, True, False

In [67]:
# ============================================================
# SECTION 9 — Query Information Extraction
# Combines signal extraction and variable extraction
# into a single unified step
# ============================================================

def extract_query_information_v2(question: str):
    """
    Extract all structured information from the user question.

    Combines two extraction strategies:
        - extract_signals()   : keyword hints for semantic reranking.
        - extract_variables() : named numeric values for rule evaluation.

    Args:
        question : Raw user question string.

    Returns:
        Tuple of (signals, variables):
            signals   — dict of matched signal labels per category.
            variables — dict of extracted named numeric values.
    """
    signals   = extract_signals(question)
    variables = extract_variables(question)
    return signals, variables

In [68]:
# ============================================================
# SECTION 10 — Decision Pipeline
# Core decision function — orchestrates all steps from
# retrieval to rule selection and output assembly
# ============================================================

def retrieve_candidate_policies(question: str, top_k: int = 5):
    """
    Retrieve top-k candidate policies via semantic similarity search.

    Encodes the question and returns the most similar policy rules
    from the pre-built embedding index.

    Args:
        question : Raw user question string.
        top_k    : Number of candidates to retrieve (default 5).

    Returns:
        List of dicts [{"score": float, "text": str}, ...]
        sorted by descending similarity score.
    """
    candidates = semantic_topk(question, k=top_k)
    return candidates


def build_decision_output(question, numbers, signals, best, candidates, constraint_used):
    """
    Assemble the final structured decision output dictionary.

    Consolidates all pipeline artifacts — the matched rule,
    confidence score, extracted variables, signals, and
    evidence candidates — into a single result object.

    Args:
        question        : Original user question string.
        numbers         : Extracted variables dict (or number list).
        signals         : Matched signal labels dict.
        best            : Best matched candidate dict {"score", "text"}.
        candidates      : Full list of retrieved candidate dicts.
        constraint_used : True if a numeric condition was matched.

    Returns:
        Dict with keys: status, question, numbers, signals,
        confidence, best_match, evidence.
    """
    return {
        "status": "ok",
        "question": question,
        "numbers": numbers,
        "signals": signals,
        "confidence": {
            "top_score": float(best["score"]),
            "constraint_used": constraint_used
        },
        "best_match": best,
        "evidence": candidates
    }


def policy_decide_v2(question: str, top_k: int = 5, min_score: float = 0.30):
    """
    Main decision function — full v2 pipeline with AND logic and priority.

    Steps:
        1. Retrieve top-k candidate policies via semantic search.
        2. Extract named variables and signals from the question.
        3. Rerank candidates using signal bonuses.
        4. Check minimum confidence threshold.
        5. Apply priority rule matching with AND condition evaluation.
        6. Return structured decision output or fallback status.

    Args:
        question  : Raw user question string.
        top_k     : Number of semantic candidates to retrieve (default 5).
        min_score : Minimum cosine similarity threshold (default 0.30).

    Returns:
        Decision result dict with keys:
            status      — "ok" | "no_candidates" | "no_confident_match"
            decision    — "NEED_MORE_INFORMATION" if fallback triggered
            best_match  — matched rule dict, or None
            variables   — extracted numeric variables
            confidence  — score and constraint_used flag
            evidence    — all retrieved candidates
    """
    q = question.strip()

    # Step 1 — Retrieve candidate policies
    candidates = retrieve_candidate_policies(q, top_k)

    if not candidates:
        return {"status": "no_candidates"}

    # Step 2 — Extract signals and named variables
    signals, variables = extract_query_information_v2(q)

    # Step 3 — Rerank candidates using signal bonuses
    candidates = rerank_with_signals(candidates, signals)

    top_score = candidates[0]["score"]

    if top_score < min_score:
        return {
            "status": "no_confident_match",
            "question": q,
            "variables": variables,
            "signals": signals,
            "confidence": {"top_score": float(top_score), "threshold": min_score},
            "best_match": None,
            "evidence": candidates
        }

    # Step 4 — Priority matching with AND condition evaluation
    best, constraint_used, fallback = apply_priority_rule_matching(candidates, variables)

    if fallback:
        return {
            "status": "ok",
            "question": q,
            "variables": variables,
            "signals": signals,
            "confidence": {"top_score": float(top_score), "constraint_used": False},
            "best_match": None,
            "decision": "NEED_MORE_INFORMATION",
            "evidence": candidates
        }

    # Step 5 — Build structured decision output
    result = build_decision_output(q, variables, signals, best, candidates, constraint_used)

    return result

In [69]:
# ============================================================
# SECTION 11 — LLM Explanation Layer
# The LLM only explains the rule engine's decision.
# It never changes or overrides the decision.
# ============================================================

def build_loan_prompt(question: str, decision_result: dict) -> str:
    """
    Build a strict prompt instructing the LLM to explain the decision.

    Extracts the matched rule, decision, and risk level from the
    decision result and formats them into a constrained prompt that
    prevents the LLM from overriding or adding new conditions.

    Args:
        question        : Original user question string.
        decision_result : Output dict from policy_decide_v2().

    Returns:
        Formatted prompt string ready for tokenization.
    """
    best = decision_result.get("best_match", {})
    rule_text  = best.get("text", "N/A")
    confidence = decision_result["confidence"]["top_score"]

    parts      = [p.strip() for p in rule_text.split("|")]
    decision   = parts[4] if len(parts) >= 5 else "UNKNOWN"
    risk       = parts[5] if len(parts) >= 6 else "UNKNOWN"

    prompt = f"""You are a loan compliance assistant.
The rule engine has already made a final decision. You must NOT change it.

User request : {question}
Matched rule : {rule_text}
Decision     : {decision}
Risk level   : {risk}
Confidence   : {confidence:.2f}

Write a short, clear explanation (2-3 sentences) telling the user:
1. What the decision is
2. Which rule was applied
3. What they should do next (if any)

Do not add any new conditions or override the decision."""

    return prompt


def llm_explain(prompt: str) -> str:
    """
    Generate a natural language explanation using the loaded LLM.

    Applies the chat template, runs generation with low temperature
    for deterministic output, and decodes only the newly generated
    tokens (skipping the prompt).

    Also removes known hallucination artifacts (e.g. "0eed") from output.

    Args:
        prompt : Formatted prompt string from build_loan_prompt().

    Returns:
        Decoded explanation string from the model.
    """
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Only decode the newly generated tokens — skip the prompt entirely
    input_len = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_len:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Remove any leftover "0eed" artifact the model hallucinates
    decoded = re.sub(r"\b0eed\b", "", decoded).strip()

    return decoded

In [70]:
# ============================================================
# SECTION 12 — Agent Wrapper
# Top-level entry point — orchestrates decide → explain → output
# ============================================================

def loan_decision_agent_v2(question: str) -> dict:
    """
    Top-level agent — runs the full decision and explanation pipeline.

    Flow:
        1. Call policy_decide_v2() for rule-based decision.
        2. Handle fallback (NEED_MORE_INFORMATION) with a static message.
        3. Handle no-match status with a generic explanation.
        4. Build LLM prompt and generate natural language explanation.
        5. Attach explanation to result and return.

    The LLM is only invoked after a confident rule match is found.
    It explains the decision — it never makes or changes it.

    Args:
        question : Raw user question string.

    Returns:
        Decision result dict with an added "explanation" key
        containing the LLM-generated natural language response.
    """
    result = policy_decide_v2(question)

    if result.get("decision") == "NEED_MORE_INFORMATION":
        result["explanation"] = (
            "Your request is missing required information. "
            "Please provide both the loan amount and your credit score."
        )
        return result

    if result["status"] != "ok":
        return {
            "status": result["status"],
            "question": question,
            "explanation": "No matching loan policy found for this request."
        }

    prompt      = build_loan_prompt(question, result)
    explanation = llm_explain(prompt)

    result["explanation"] = explanation
    return result

In [71]:
# ============================================================
# SECTION 13 — Test Cases & Evaluation
# 18 test cases covering all decision branches:
#   Low risk (5), Medium risk (5), High risk (5), Fallback (3)
# ============================================================

test_cases_v2 = [
    # LOW RISK — amount <= 5000 (5 cases)
    "I need a loan of 1000 dollars. My credit score is 750.",
    "Apply for a 2500 loan. Credit score is 800.",
    "Loan request for 4000. My credit rating is 720.",
    "Can I borrow 5000? Credit score 690.",
    "I want a 3000 loan. My score is 650.",

    # MEDIUM RISK — 5000 < amount <= 20000 (5 cases)
    "I need a loan of 8000 dollars. My credit score is 710.",
    "Requesting 12000. Credit score is 680.",
    "Apply for 15000 loan. My credit score is 740.",
    "I want to borrow 20000. Credit score 720.",
    "Loan of 9500. My credit rating is 760.",

    # HIGH RISK — amount > 20000 AND credit < 700, or credit < 600 (5 cases)
    "I want a loan of 25000 dollars. My credit score is 680.",
    "Requesting 30000 loan. Credit score is 650.",
    "Apply for 50000. My credit score is 580.",
    "I need 22000. Credit score is 590.",
    "Loan of 100000. My credit rating is 520.",

    # MISSING VARIABLE — fallback cases (3 cases)
    "I need a loan.",
    "My credit score is 700.",
    "I want to borrow some money for my business.",
]

print("\n" + "="*60)
print("LOAN DECISION AGENT v2 — TEST RESULTS")
print("="*60)

for q in test_cases_v2:
    print(f"\nQ: {q}")
    out = loan_decision_agent_v2(q)
    print(f"Status    : {out['status']}")

    decision = out.get("decision", "N/A")
    if out["status"] == "ok" and out.get("best_match"):
        parts    = [p.strip() for p in out["best_match"]["text"].split("|")]
        decision = parts[4] if len(parts) >= 5 else "N/A"
        risk     = parts[5] if len(parts) >= 6 else "N/A"
        print(f"Decision  : {decision}")
        print(f"Risk      : {risk}")
        print(f"Variables : {out.get('numbers', out.get('variables', {}))}")
        print(f"Confidence: {out['confidence']['top_score']:.2f}")
    else:
        print(f"Decision  : {decision}")
        print(f"Variables : {out.get('variables', {})}")

    print(f"Explain   : {out.get('explanation', '')}")
    print("-"*60)



LOAN DECISION AGENT v2 — TEST RESULTS

Q: I need a loan of 1000 dollars. My credit score is 750.
Status    : ok
Decision  : APPROVED
Risk      : Low
Variables : {'amount': 1000.0, 'credit_score': 750.0}
Confidence: 0.62
Explain   : The decision to approve your loan request has been made based on the "Small loan auto approved" rule, which applies to loans up to $5000 with a low risk level. Since your request of $1000 falls within this range and your credit score is 750, your loan has been approved with a confidence level of . You should proceed with the next steps to secure the loan, such as completing the necessary paperwork and finalizing the loan agreement.
------------------------------------------------------------

Q: Apply for a 2500 loan. Credit score is 800.
Status    : ok
Decision  : APPROVED
Risk      : Low
Variables : {'amount': 2500.0, 'credit_score': 800.0}
Confidence: 0.52
Explain   : The decision is Approved for your loan application, based on the rule that applies to S

In [1]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


In [ ]:
!pip install bitsandbytes accelerate

In [3]:
# -------------------------
# 0) Setup & Imports
# -------------------------
import json
import re
import torch
import pandas as pd
from typing import Dict, Any, Tuple, Optional, List

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# -------------------------
# 1) Load Model (same approach)
# -------------------------
# Change these paths to match your environment
model_path = "/content/drive/MyDrive/Phi_3_5_mini_instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    local_files_only=True
)

print("Model & tokenizer loaded")

Device: cuda


This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Model & tokenizer loaded


In [25]:

loan_rules_v2 = [
    ["section", "policy_name", "rule_description", "condition", "approval_required", "risk_level"],
    ["Loan", "Small Loan",           "Small loan auto approved",           "amount <= 5000",                              "APPROVED",             "Low"],
    ["Loan", "Medium Loan",          "Medium loan review",                 "5000 < amount <= 20000",                      "MANUAL_REVIEW",        "Medium"],
    ["Loan", "High Loan Low Credit", "High loan with low credit rejected", "amount > 20000 AND credit_score < 700",       "REJECTED",             "High"],
    ["Loan", "Very Low Credit",      "Very low credit score rejection",    "credit_score < 600",                          "REJECTED",             "High"],
]

loan_csv_v2 = "Loan_Policy_v2.csv"

with open(loan_csv_v2, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(loan_rules_v2)

print("Loan v2 CSV created.")

# -------------------------
# Reload pipeline with v2 CSV
# (same Section 2 block — only csv path changes)
# -------------------------

Policy_Csv = loan_csv_v2

df = pd.read_csv(Policy_Csv)

cols = ["section", "policy_name", "rule_description",
        "condition", "approval_required", "risk_level"]

df[cols] = df[cols].fillna("")

df["full_text"] = df.apply(lambda r: clean_text(
    f"{r['section']} | {r['policy_name']} | "
    f"{r['rule_description']} | {r['condition']} | "
    f"{r['approval_required']} | {r['risk_level']}"
), axis=1)

policy_texts = df["full_text"].tolist()
print("Loan v2 rules loaded:", len(policy_texts))
for t in policy_texts:
    print(" ", t)


Loan v2 CSV created.
Loan v2 rules loaded: 4
  Loan | Small Loan | Small loan auto approved | amount <= 5000 | APPROVED | Low
  Loan | Medium Loan | Medium loan review | 5000 < amount <= 20000 | MANUAL_REVIEW | Medium
  Loan | High Loan Low Credit | High loan with low credit rejected | amount > 20000 AND credit_score < 700 | REJECTED | High
  Loan | Very Low Credit | Very low credit score rejection | credit_score < 600 | REJECTED | High


In [27]:
#-----------------------
# Semantic Embedding Setup
# -------------------------
# Load sentence transformer model for semantic search

from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
policy_emb = embedder.encode(
    policy_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Loan embeddings ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loan embeddings ready.


In [41]:
def semantic_topk(query: str, k: int = 5):
    """
    Retrieve top-k most similar policies using cosine similarity.
    """
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = (policy_emb @ q_emb.T).squeeze()

    idx = np.argsort(scores)[-k:][::-1]

    return [
        {"score": float(scores[i]), "text": policy_texts[i]}
        for i in idx
    ]

In [42]:
SIGNAL_DICT = {
    "amounts": {
        "small":  ["small", "little", "minor", "1000", "2000", "3000", "4000", "5000"],
        "medium": ["medium", "moderate", "12000", "15000", "8000", "9000", "10000"],
        "large":  ["large", "big", "huge", "25000", "30000", "50000", "100000"]
    },

    "credit": {
        "excellent": ["excellent credit", "great credit", "800", "850"],
        "good":      ["good credit", "700", "710", "720", "740", "750", "760"],
        "poor":      ["bad credit", "low credit", "poor credit", "580", "590", "600", "650", "680"]
    },

    "actions": {
        "apply":    ["apply", "applying", "application"],
        "borrow":   ["borrow", "borrowing", "loan", "request", "need"]
    }
}

In [43]:
def keyword_match(text, keywords):
    """
    Match keywords using word boundaries.
    Avoid false substring matches.
    """
    for kw in keywords:
        if re.search(rf"\b{re.escape(kw)}\b", text):
            return True
    return False

In [44]:
# -------------------------
# 7) Numeric Extraction & Rule Matching
# -------------------------
import re

def extract_numbers(q: str):
    """
    Extract numeric values from user query.
    Useful for price-based policies.
    """
    nums = re.findall(r"\d{1,9}(?:,\d{9})*(?:\.\d+)?", q)
    return [float(n.replace(",", "")) for n in nums]

def get_condition_from_rule_text(rule_text: str) -> str:
    # expected format: section | policy | rule_desc | condition | approval | risk
    parts = [p.strip() for p in rule_text.split("|")]
    return parts[3].strip() if len(parts) >= 4 else ""

def match_numeric_condition(cond: str, value: float) -> bool:
    cond = cond.strip()
    if not cond:
        return False

    # chained: 1500 < amount <= 3000
    m = re.match(r"^\s*([0-9\.]+)\s*<\s*\w+\s*<=\s*([0-9\.]+)\s*$", cond)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
        return lo < value <= hi

    # simple: amount > 3000  OR payment_terms_days <= 60
    m = re.match(r"^\s*\w+\s*(<=|>=|<|>|==)\s*([0-9\.]+)\s*$", cond)
    if m:
        op, rhs = m.group(1), float(m.group(2))
        if op == "<=": return value <= rhs
        if op == ">=": return value >= rhs
        if op == "<":  return value < rhs
        if op == ">":  return value > rhs
        if op == "==": return value == rhs

    return False

In [45]:
def retrieve_candidate_policies(question: str, top_k: int = 5):
    """
    Step 1 — Retrieve top candidate policies using semantic search.

    The embedding model compares the user question with
    policy embeddings and returns the most similar rules.
    """

    candidates = semantic_topk(question, k=top_k)

    return candidates

In [52]:
def extract_signals(q: str):

    ql = q.lower()
    signals = {category: [] for category in SIGNAL_DICT.keys()}

    return signals

In [54]:
def rerank_with_signals(cands, signals, bonus=0.08):
    out = []
    for c in cands:
        text_l = c["text"].lower()
        score = c["score"]

        for category, labels in signals.items():
            for label in labels:
                if label in text_l:
                    score += bonus

        out.append({"score": float(score), "text": c["text"]})

    out.sort(key=lambda x: x["score"], reverse=True)
    return out

In [48]:
def build_decision_output(question, numbers, signals, best, candidates, constraint_used):
    """
    Step 5 — Build a structured decision object.

    This object contains:
    - selected rule
    - confidence score
    - numeric values
    - evidence policies
    """

    return {
        "status": "ok",
        "question": question,
        "numbers": numbers,
        "signals": signals,
        "confidence": {
            "top_score": float(best["score"]),
            "constraint_used": constraint_used
        },
        "best_match": best,
        "evidence": candidates
    }

In [49]:
def policy_decide(question: str, top_k: int = 5, min_score: float = 0.30):

    q = question.strip()

    # Step 1 — Retrieve candidate policies
    candidates = retrieve_candidate_policies(q, top_k)

    if not candidates:
        return {"status": "no_candidates"}

    # Step 2 — Extract structured query information
    numbers = extract_numbers(q)

    top_score = candidates[0]["score"]

    if top_score < min_score:
        return {
            "status": "no_confident_match",
            "question": q,
            "numbers": numbers,
            "confidence": {
                "top_score": float(top_score),
                "threshold": min_score
            },
            "best_match": None,
            "evidence": candidates
        }

    # Step 4 — Apply numeric rule matching
    best, constraint_used = apply_numeric_rule_matching(candidates, numbers)

    # Step 5 — Build structured decision output
    result = build_decision_output(
        q,
        numbers,
        best,
        candidates,
        constraint_used
    )

    return result

In [50]:
# -------------------------
# Step 5 — LLM Explanation Layer
# (wraps policy_decide — LLM only explains, never decides)
# -------------------------

def build_loan_prompt(question: str, decision_result: dict) -> str:
    """
    Build a strict prompt so the LLM explains the decision
    without changing it.
    """
    best = decision_result.get("best_match", {})
    rule_text  = best.get("text", "N/A")
    confidence = decision_result["confidence"]["top_score"]

    parts      = [p.strip() for p in rule_text.split("|")]
    decision   = parts[4] if len(parts) >= 5 else "UNKNOWN"
    risk       = parts[5] if len(parts) >= 6 else "UNKNOWN"

    prompt = f"""You are a loan compliance assistant.
The rule engine has already made a final decision. You must NOT change it.

User request : {question}
Matched rule : {rule_text}
Decision     : {decision}
Risk level   : {risk}
Confidence   : {confidence:.2f}

Write a short, clear explanation (2-3 sentences) telling the user:
1. What the decision is
2. Which rule was applied
3. What they should do next (if any)

Do not add any new conditions or override the decision."""

    return prompt


def llm_explain(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Only decode the newly generated tokens — skip the prompt entirely
    input_len = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_len:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Remove any leftover "0eed" artifact the model hallucinates
    decoded = re.sub(r"\b0eed\b", "", decoded).strip()

    return decoded


In [55]:
# -------------------------
# Step 2: Extract Multiple Named Variables
# Extended from extract_numbers() — adds keyword detection
# -------------------------

# Keyword map: variable name → trigger words to look for in question
VARIABLE_KEYWORDS = {
    "amount":       ["loan", "borrow", "request", "need", "apply", "amount"],
    "credit_score": ["credit", "score", "fico", "rating"]
}

def extract_variables(q: str) -> dict:
    """
    Extract named numeric variables from user input.
    Uses keyword detection + extract_numbers() from Section 5.

    Strategy:
    - Scan sentence by sentence
    - If a sentence contains a variable keyword, assign
      the first number found in that sentence to that variable
    - Falls back to positional assignment if keyword not found

    Returns dict e.g. {"amount": 25000.0, "credit_score": 680.0}
    """

    ql = q.lower()
    result = {}

    # Split into sentences for localized keyword matching
    sentences = re.split(r"[.,;]|\band\b", ql)

    for var, keywords in VARIABLE_KEYWORDS.items():
        for sentence in sentences:
            if any(kw in sentence for kw in keywords):
                # Reuse extract_numbers on this sentence fragment
                nums = extract_numbers(sentence)
                if nums:
                    result[var] = nums[0]
                    break   # first matching sentence wins

    return result


# -------------------------
# Level 2 — Step 3: AND Condition Matching
# Extended from match_numeric_condition() — adds AND split + variable substitution
# -------------------------

def match_and_condition(condition: str, variables: dict) -> bool:
    """
    Evaluate a condition string that may contain AND clauses.
    Uses match_numeric_condition() from Section 5 for each part.
    No eval(). Safe manual evaluation.

    Supports:
        amount <= 5000
        5000 < amount <= 20000
        amount > 20000 AND credit_score < 700
        credit_score < 600
    """

    # Split on AND — gives list of individual conditions
    parts = [p.strip() for p in re.split(r"\bAND\b", condition, flags=re.IGNORECASE)]

    for part in parts:

        # Detect which variable this part refers to
        matched_var = None
        matched_val = None

        for var, value in variables.items():
            if var in part:
                matched_var = var
                matched_val = value
                break

        # Variable referenced in condition but not supplied by user → fail this part
        if matched_var is None:
            return False

        # Normalize: replace variable name with placeholder "amount"
        # so match_numeric_condition() (which uses \w+) can parse it
        normalized = re.sub(rf"\b{re.escape(matched_var)}\b", "amount", part)

        # Reuse match_numeric_condition from Section 5
        if not match_numeric_condition(normalized, matched_val):
            return False

    # All parts passed
    return True


# -------------------------
# Level 2 — Step 4: Priority Strategy
# Replaces apply_numeric_rule_matching — adds ALL-rules evaluation + priority sort
# -------------------------

RISK_PRIORITY = {"High": 3, "Medium": 2, "Low": 1}

def apply_priority_rule_matching(candidates: list, variables: dict):
    """
    Extended from apply_numeric_rule_matching().

    Changes:
    - Evaluates ALL candidates (not just stopping at first match)
    - Applies AND condition logic via match_and_condition()
    - Sorts matched rules by risk priority (High > Medium > Low)
    - Returns fallback status if no rule matches or variables missing

    Returns: (best_candidate, constraint_used, fallback)
    """

    if not variables:
        # No variables extracted — cannot evaluate any numeric condition
        return candidates[0], False, True

    matched_rules = []
    unmatched_rules = []

    for c in candidates:
        cond = get_condition_from_rule_text(c["text"])

        # Extract risk level from rule text (6th pipe-separated field)
        parts = [p.strip() for p in c["text"].split("|")]
        risk  = parts[5] if len(parts) >= 6 else "Low"

        has_numeric = bool(re.search(r"\w+\s*(<=|>=|<|>|==)\s*\d", cond)) or \
                      bool(re.search(r"\d+\s*<\s*\w+\s*<=\s*\d+", cond))

        if has_numeric:
            if match_and_condition(cond, variables):
                matched_rules.append((RISK_PRIORITY.get(risk, 0), c["score"], risk, c))
            else:
                unmatched_rules.append(c)
        else:
            unmatched_rules.append(c)

    if not matched_rules:
        # No rule matched — trigger fallback
        return candidates[0], False, True

    # Sort: highest risk priority first, then semantic score as tiebreaker
    matched_rules.sort(key=lambda x: (x[0], x[1]), reverse=True)

    _, _, _, best = matched_rules[0]
    return best, True, False


# -------------------------
# Extended extract_query_information for Level 2
# Returns signals + named variables dict (instead of flat number list)
# -------------------------

def extract_query_information_v2(question: str):
    """
    Extended from extract_query_information().
    Now returns named variables dict instead of flat number list.
    signals  — keyword hints for reranking (unchanged)
    variables — {"amount": 25000, "credit_score": 680}
    """
    signals   = extract_signals(question)       # reused from Section 4
    variables = extract_variables(question)     # new Level 2 function
    return signals, variables



# -------------------------
# Level 2 — Agent wrapper
# Same structure as loan_decision_agent() — only calls v2 decide
# -------------------------

def loan_decision_agent_v2(question: str) -> dict:
    """
    Same structure as loan_decision_agent().
    Calls policy_decide_v2 instead of policy_decide.
    """

    result = policy_decide_v2(question)

    if result.get("decision") == "NEED_MORE_INFORMATION":
        result["explanation"] = (
            "Your request is missing required information. "
            "Please provide both the loan amount and your credit score."
        )
        return result

    if result["status"] != "ok":
        return {
            "status": result["status"],
            "question": question,
            "explanation": "No matching loan policy found for this request."
        }

    prompt      = build_loan_prompt(question, result)   # reused from Level 1
    explanation = llm_explain(prompt)                   # reused from Level 1

    result["explanation"] = explanation
    return result


# -------------------------
# Step 6 — 18 Test Cases
# -------------------------

test_cases_v2 = [
    # LOW RISK — amount <= 5000 (5 cases)
    "I need a loan of 1000 dollars. My credit score is 750.",
    "Apply for a 2500 loan. Credit score is 800.",
    "Loan request for 4000. My credit rating is 720.",
    "Can I borrow 5000? Credit score 690.",
    "I want a 3000 loan. My score is 650.",

    # MEDIUM RISK — 5000 < amount <= 20000 (5 cases)
    "I need a loan of 8000 dollars. My credit score is 710.",
    "Requesting 12000. Credit score is 680.",
    "Apply for 15000 loan. My credit score is 740.",
    "I want to borrow 20000. Credit score 720.",
    "Loan of 9500. My credit rating is 760.",

    # HIGH RISK — amount > 20000 AND credit < 700, or credit < 600 (5 cases)
    "I want a loan of 25000 dollars. My credit score is 680.",
    "Requesting 30000 loan. Credit score is 650.",
    "Apply for 50000. My credit score is 580.",
    "I need 22000. Credit score is 590.",
    "Loan of 100000. My credit rating is 520.",

    # MISSING VARIABLE — fallback cases (3 cases)
    "I need a loan.",
    "My credit score is 700.",
    "I want to borrow some money for my business.",
]

print("\n" + "="*60)
print("LOAN DECISION AGENT v2 — TEST RESULTS")
print("="*60)

for q in test_cases_v2:
    print(f"\nQ: {q}")
    out = loan_decision_agent_v2(q)
    print(f"Status    : {out['status']}")

    decision = out.get("decision", "N/A")
    if out["status"] == "ok" and out.get("best_match"):
        parts    = [p.strip() for p in out["best_match"]["text"].split("|")]
        decision = parts[4] if len(parts) >= 5 else "N/A"
        risk     = parts[5] if len(parts) >= 6 else "N/A"
        print(f"Decision  : {decision}")
        print(f"Risk      : {risk}")
        print(f"Variables : {out.get('numbers', out.get('variables', {}))}")
        print(f"Confidence: {out['confidence']['top_score']:.2f}")
    else:
        print(f"Decision  : {decision}")
        print(f"Variables : {out.get('variables', {})}")

    print(f"Explain   : {out.get('explanation', '')}")
    print("-"*60)


LOAN DECISION AGENT v2 — TEST RESULTS

Q: I need a loan of 1000 dollars. My credit score is 750.
Status    : ok
Decision  : APPROVED
Risk      : Low
Variables : {'amount': 1000.0, 'credit_score': 750.0}
Confidence: 0.54
Explain   : The decision is that your loan request for $1000 has been approved based on the "Small Loan" rule that applies to loans with an amount less than or equal to $5000. Given the low risk level and the confidence of 0.54, you can proceed with the application process as recommended. There are no further actions required from your side at this moment.
------------------------------------------------------------

Q: Apply for a 2500 loan. Credit score is 800.
Status    : ok
Decision  : APPROVED
Risk      : Low
Variables : {'amount': 2500.0, 'credit_score': 800.0}
Confidence: 0.52
Explain   : The loan application for a small auto loan has been approved based on the rule that applies to loans with an amount less than or equal to $5000, which matches your request for 